In [1]:
#!pip install neo4j

In [2]:
from neo4j import GraphDatabase

url = "neo4j://127.0.0.1:7687"
username = "neo4j"
password = "password"

driver = GraphDatabase.driver(url, auth=(username, password))

In [11]:
# add data to the graph
def create_knowledge_graph(tx):
    query = """
    CREATE (p:PERSON {name: "John Doe", age: 30})
    CREATE (c:COMPANY {name: "Acme Inc", industry: "Technology"})
    CREATE (p)-[:WORKS_FOR]->(c)
    CREATE (p2:PERSON {name: "Jane Smith", age: 25})
    CREATE (p2) - [:WORKS_FOR]->(c)
    CREATE (s:SKILL {name: "Python"})
    CREATE (p)-[:HAS_SKILL]->(s)
    CREATE (p2)-[:HAS_SKILL]->(s)
    CREATE (p3:PERSON {name: "Alice Johnson", age: 35})
    CREATE (p3)-[:KNOWS]->(p)
    RETURN "Knowledge graph created successfully"
    """
    tx.run(query)

In [12]:
# query the graph
def query_graph(tx):
    query = """
    MATCH (p:PERSON)-[w:WORKS_FOR]->(c:COMPANY)
    RETURN p.name as person, c.name as company, p.age as age
    """
    result = tx.run(query)
    return [row for row in result]

In [13]:
# tx - transaction object (it is a group of operations, they are atomic)
# session - the group of transactions
# driver - the connection to the database

with driver.session() as session:
    session.execute_write(create_knowledge_graph)
    data = session.execute_read(query_graph)

In [18]:
# print the retrieved data
for record in data:
    print(f"{record['person']} {record['age']} years old, works at {record['company']}.")

John Doe 30 years old, works at Acme Inc.
Jane Smith 25 years old, works at Acme Inc.


In [19]:
# close the driver
driver.close()